# USGS Water — annual peaks and rating curves

The site-keyed services: annual peak streamflow and the stage-discharge
rating curve for a gauge. Both require an explicit `sites=`.

## What this notebook does

1. **Annual peaks** — the highest instantaneous streamflow recorded each
   year at a single gauge (USGS Water is a *tabular* backend → `download()`
   writes a CSV and returns a `DataFrame`).
2. **Rating curve** — the stage-discharge relationship for the same gauge,
   plotted with matplotlib.

> No credentials needed — USGS Water is a public service. Both services are
> keyed on a single gauge id passed via `sites=`.

## Setup

A single import cell: `matplotlib` for the rating-curve plot and the
unified `EarthLens` entry point.

In [ ]:
import matplotlib.pyplot as plt
from earthlens.earthlens import EarthLens

## 1 · Annual peak streamflow

The `peaks` service returns one row per water year — the annual maximum
instantaneous discharge at gauge `01646500` (Potomac River near
Washington, D.C.). We build the request first: source, variable, the date
window, the gauge id, and the `legacy` API plane that serves this service.

In [ ]:
peaks_request = EarthLens(
    data_source="usgs-water",
    variables=["discharge"],
    start="1950-01-01",
    end="2020-12-31",
    aoi=[-77.2, 38.9, -77.0, 39.0],
    path="_nb_out",
    service="peaks",
    sites="01646500",
    api="legacy",
)

### Download the peaks

`download()` fetches the annual peaks, writes them to a CSV under
`_nb_out`, and returns the same rows as a `DataFrame`.

In [ ]:
peaks = peaks_request.download(progress_bar=False)
peaks.head()

## 2 · Stage-discharge rating curve

The `ratings` service returns the rating table that converts gauge stage
(water height) into discharge for the same gauge. The date window is
incidental here — the rating curve is a single current relationship — so a
short window keeps the request small.

In [ ]:
ratings_request = EarthLens(
    data_source="usgs-water",
    variables=["discharge"],
    start="2023-01-01",
    end="2023-01-05",
    aoi=[-77.2, 38.9, -77.0, 39.0],
    path="_nb_out",
    service="ratings",
    sites="01646500",
    api="legacy",
)

### Download the rating table

As before, `download()` writes the CSV and hands back the `DataFrame`.

In [ ]:
ratings = ratings_request.download(progress_bar=False)

### Plot the rating curve

Stage on the x-axis against discharge on the y-axis gives the classic
rising rating curve.

In [ ]:
ax = ratings.plot(x="stage", y="discharge", legend=False, figsize=(6, 4))
ax.set_xlabel("Stage (ft)")
ax.set_ylabel("Discharge (ft3/s)")
ax.set_title("Stage-discharge rating curve")
plt.tight_layout()